In [6]:
import pandas as pd

In [7]:
lasso = pd.read_csv(
    "../submissions/linear_submission.csv"
)

xgb = pd.read_csv(
    "../submissions/xgb_random_search_submission.csv"
)

In [8]:
import numpy as np

# =====================================================================
# 1. 宣告最佳化黃金權重 (由 SciPy SLSQP 求解 5-Fold OOF 預測值得出)
# =====================================================================
W_LASSO = 0.50071
W_XGB = 0.49929

# =====================================================================
# 2. 執行對數空間幾何加權平均
# =====================================================================
lasso_log = np.log1p(lasso["SalePrice"])
xgb_log = np.log1p(xgb["SalePrice"])

ensemble = pd.DataFrame({
    "Id": lasso["Id"],
    "SalePrice": np.expm1((W_LASSO * lasso_log) + (W_XGB * xgb_log)),
})

ensemble.head()

,Id,SalePrice
0,1461,120741.880409
1,1462,162965.948956
2,1463,182430.723453
3,1464,198154.364097
4,1465,187837.759707


In [9]:
ensemble.to_csv(
    "../submissions/ensemble_submission.csv",
    index=False
)

print("Ensemble Submission 已建立")

Ensemble Submission 已建立


In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.linear_model import Lasso, Ridge, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_predict
from xgboost import XGBRegressor

# =========================================================
# 第 1 步：讀取訓練與測試資料
# =========================================================
print("1. 載入訓練與測試資料中...")
# 線性模型 (Lasso, Ridge) 使用經過縮放與剔除共線性特徵的資料
X_train_linear = pd.read_csv("../data/X_train.csv")  
X_test_linear = pd.read_csv("../data/X_test.csv")

# 樹狀模型 (XGB, RF) 使用原始保留完整特徵的資料
X_train_tree = pd.read_csv("../data/X_train_tree.csv")  
X_test_tree = pd.read_csv("../data/X_test_tree.csv")

# 真實房價 (Log空間) 與測試集 Id
y_train = pd.read_csv("../data/y_train.csv").squeeze()  
test_ids = pd.read_csv("../data/test.csv")["Id"]

# =========================================================
# 第 2 步：定義四大基礎模型 (參數依據你們先前的最佳實驗結果)
# =========================================================
lasso_model = Lasso(alpha=0.0005, max_iter=50000, random_state=42)
ridge_model = Ridge(alpha=10.0)
xgb_model = XGBRegressor(
    n_estimators=3500, learning_rate=0.015, max_depth=3,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=2,
    gamma=0.01, reg_alpha=0.01, reg_lambda=1.5, random_state=42, n_jobs=-1
)
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

# =========================================================
# 第 3 步：自動計算四大基礎模型的 5-Fold OOF 預測值
# =========================================================
print("2. 正在計算 四大模型 5-Fold OOF 預測值 (大約需時 15~30 秒)...")
oof_lasso = cross_val_predict(lasso_model, X_train_linear, y_train, cv=5)
oof_ridge = cross_val_predict(ridge_model, X_train_linear, y_train, cv=5)
oof_xgb = cross_val_predict(xgb_model, X_train_tree, y_train, cv=5, n_jobs=-1)
oof_rf = cross_val_predict(rf_model, X_train_tree, y_train, cv=5, n_jobs=-1)

# =========================================================
# 第 4 步：(精華改進！) 訓練 Stacking Meta-Model (元模型)
# =========================================================
print("3. 正在執行 Stacking 訓練元模型 (Meta-Model)...")

# 將四個模型的 OOF 預測值組合起來，變成第二階段的新特徵矩陣
X_meta_train = np.column_stack([oof_lasso, oof_ridge, oof_xgb, oof_rf])

# 使用 RidgeCV 作為元模型，它會自動透過交叉驗證尋找最完美的融合係數
meta_model = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
meta_model.fit(X_meta_train, y_train)

# 印出 Meta-Model 幫這四個模型分配的影響力係數 (可以放進專題報告中！)
print("=" * 45)
print("【Stacking 元模型訓練完成！】")
print(f"Lasso 影響力係數: {meta_model.coef_[0]:.5f}")
print(f"Ridge 影響力係數: {meta_model.coef_[1]:.5f}")
print(f"XGB   影響力係數: {meta_model.coef_[2]:.5f}")
print(f"RF    影響力係數: {meta_model.coef_[3]:.5f}")
print(f"Meta-Model 截距 (Bias): {meta_model.intercept_:.5f}")
print("=" * 45 + "\n")

# =========================================================
# 第 5 步：使用全體訓練集訓練基礎模型，並預測測試集
# =========================================================
print("4. 正在以全體資料訓練基礎模型並產生預測...")
lasso_model.fit(X_train_linear, y_train)
ridge_model.fit(X_train_linear, y_train)
xgb_model.fit(X_train_tree, y_train)
rf_model.fit(X_train_tree, y_train)

pred_lasso = lasso_model.predict(X_test_linear)
pred_ridge = ridge_model.predict(X_test_linear)
pred_xgb = xgb_model.predict(X_test_tree)
pred_rf = rf_model.predict(X_test_tree)

# =========================================================
# 第 6 步：將測試集預測丟給 Meta-Model 進行最終融合並產出 Submission
# =========================================================
print("5. 正在透過元模型進行最終決策...")
# 把測試集的四個預測結果組合起來，變成 Meta-Model 的測試特徵
X_meta_test = np.column_stack([pred_lasso, pred_ridge, pred_xgb, pred_rf])

# 讓 Meta-Model 進行最終融合預測 (此時仍在 Log 空間)
ensemble_log_stack = meta_model.predict(X_meta_test)

# 使用 np.expm1 將對數還原為真實房價金額 (專案核心規範)
stacking_submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": np.expm1(ensemble_log_stack)
})

os.makedirs("../submissions", exist_ok=True)
output_path = "../submissions/stacking_4models_submission.csv"
stacking_submission.to_csv(output_path, index=False)

print(f"四模型 Stacking 預測檔已存為 {output_path}")

1. 載入訓練與測試資料中...
2. 正在計算 四大模型 5-Fold OOF 預測值 (大約需時 15~30 秒)...
3. 正在最佳化模型融合權重...
【四大模型優化求解完成！】
Lasso  最優權重: 0.54293
Ridge  最優權重: 0.00000
XGB    最優權重: 0.45707
RF     最優權重: 0.00000
混合後本機 CV OOF RMSE: 0.10771

4. 正在以全體資料訓練模型並產生最終預測...
四大模型預測檔已存為 ../submissions/ensemble_4models_submission.csv
